# Comparing Alignments

`WLCM-BSB-manualnew.json` works for Rick's internlinear process, i think.
`WLCM-BSB-manual21.json` is updated to the 0.2.1 Alignment standard. 
Both have 842 missing target tokens, and 4 missing source tokens. 

In [1]:
from src.check import AlignmentPairs
from src.burrito import DATAPATH, Manager, AlignmentSet
LANGDATAPATH = DATAPATH.parent.parent / "alignments-eng/data"
# "new" = newer data, but not fixed
alsetnew = AlignmentSet(targetlanguage="eng", targetid="BSB", sourceid="WLCM", alternateid="manualnew", langdatapath=LANGDATAPATH)
# keep the bad records so we can compare against `manual21`
mgrnew = Manager(alsetnew, keepbadrecords=True)
# "21" = updated for 0.2.1 spec, drops the bad records
alset21 = AlignmentSet(targetlanguage="eng", targetid="BSB", sourceid="WLCM", alternateid="manual21", langdatapath=LANGDATAPATH)
mgr21 = Manager(alset21)



        - sourcepath: /Users/sboisen/git/Clear-Bible/internal-Alignments/data/sources/WLCM.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/targets/BSB/ot_BSB.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/WLCM-BSB-manualnew.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-eng/data/alignments/BSB/WLCM-BSB-manualnew.toml
        
No source selectors for 08003012.005: dropping the record, adding to self.badrecords.
No source selectors for 12005018.023: dropping the record, adding to self.badrecords.
No source selectors for 19021001.011: dropping the record, adding to self.badrecords.
No source selectors for 24039012.009: dropping the record, adding to self.badrecords.
Keeping 842 bad alignment records. Instances in self.badrecords.
Token reference is missing from self.targetitems	838
No source selectors	4
23213 BCV sources < 23214 records.

        - sourcepath: /Users/sboisen/git/Clear-Bible/i

## Looking at differences

`mgr21` has 22 fewer BCVs than `mgrnew`:
* The first case is a record with no source selectors: so indexing by BCV returns an empty string. This case should now be better logged in self.badrecords.
* The other cases are issues in Psalms


In [2]:
apairs = AlignmentPairs.AlignmentPairs(mgrnew, mgr21)

/Users/sboisen/git/Clear-Bible/internal-Alignments/src/check/AlignmentPairs.py:19: UserWarning: Managers are different lengths: likely a problem.
23214, 23192
  warn(f"Managers are different lengths: likely a problem.\n{len(self.manager1)}, {len(self.manager2)}")
/Users/sboisen/git/Clear-Bible/internal-Alignments/src/check/AlignmentPairs.py:20: UserWarning: See self.bcv_diffs for the non-equal cases.
  warn("See self.bcv_diffs for the non-equal cases.")


In [3]:
print(f"{len(apairs.bcv_diffs)} BCV differences")
for diff in apairs.bcv_diffs: print(diff)

22 BCV differences
- 
- 19005002
- 19006002
- 19007002
- 19009002
- 19018002
- 19034002
- 19046002
- 19051003
- 19052003
- 19054003
- 19056002
- 19058002
- 19059002
- 19060002
- 19060003
- 19069002
- 19076002
- 19077002
- 19084002
- 19088002
- 19102002


In [7]:
# the first diff is an empty BCV value from a record with no source selectors
# there would be four of these, but each get overwritten in the dict with a "" key
mgrnew.bcv["records"][""]

[<AlignmentRecord: {'source': <WLCM: []>, 'target': <BSB: ['24039012022']>}>]

In [8]:
mgrnew.bcv["records"][""][0].meta

Metadata(status='created' id='24039012.009' origin='manual')

In [12]:
mgrnew.badrecords["24039012.009"]

<BadRecord (24039012.009): 'No source selectors'>

## Other BCV Differences

What about the diffs from the Psalms? Let's look at 19005002.

In [17]:
"19005002" in mgrnew.bcv["records"]

True

In [18]:
# but it's not in mgr21, presumably because it was dropped when the earlier version was loaded
"19005002" in mgr21.bcv["records"]

False

In [19]:
mgrnew.bcv["records"]["19005002"]

[<AlignmentRecord: {'source': <WLCM: ['o190050020011', 'o190050020012']>, 'target': <BSB: ['19005001018', '19005001019', '19005001020']>}>,
 <AlignmentRecord: {'source': <WLCM: ['o190050020021', 'o190050020022']>, 'target': <BSB: ['19005001016', '19005001017']>}>,
 <AlignmentRecord: {'source': <WLCM: ['o190050020031']>, 'target': <BSB: ['19005001022', '19005001023']>}>,
 <AlignmentRecord: {'source': <WLCM: ['o190050020041', 'o190050020042']>, 'target': <BSB: ['19005001025']>}>,
 <AlignmentRecord: {'source': <WLCM: ['o190050020051', 'o190050020052']>, 'target': <BSB: ['19005001026', '19005001027']>}>]

In [34]:
# 5 records for PSA 5:2 are all bad, so the BCV gets dropped altogether, and therefore shows up in apairs.bcv_diffs
bcvid = "19005002"
for rec in mgrnew.bcv["records"][bcvid]:
    recid = rec.meta.id
    print(f"{recid}: badrecord? {recid in mgrnew.badrecords}")
    if recid in mgrnew.badrecords:
        print(mgrnew.badrecords[recid])

19005001.006: badrecord? True
<BadRecord (19005001.006): 'Token reference is missing from self.targetitems', ['19005001018', '19005001019', '19005001020']>
19005001.007: badrecord? True
<BadRecord (19005001.007): 'Token reference is missing from self.targetitems', ['19005001016', '19005001017']>
19005001.008: badrecord? True
<BadRecord (19005001.008): 'Token reference is missing from self.targetitems', ['19005001022', '19005001023']>
19005001.009: badrecord? True
<BadRecord (19005001.009): 'Token reference is missing from self.targetitems', ['19005001025']>
19005001.010: badrecord? True
<BadRecord (19005001.010): 'Token reference is missing from self.targetitems', ['19005001026', '19005001027']>


In [36]:
# are _all_ these token references missing, or just some?
# yes ALL of them
for rec in mgrnew.bcv["records"][bcvid]:
    for targettok in rec.target_selectors:
        if targettok not in mgrnew.targetitems: print(targettok)

19005001018
19005001019
19005001020
19005001016
19005001017
19005001022
19005001023
19005001025
19005001026
19005001027


In [24]:
# Same check for PSA 6:2: same result
for rec in mgrnew.bcv["records"]["19006002"]:
    recid = rec.meta.id
    print(f"{recid}: badrecord? {recid in mgrnew.badrecords}")

19006001.007: badrecord? True
19006001.008: badrecord? True
19006001.009: badrecord? True
19006001.010: badrecord? True
19006001.011: badrecord? True
19006001.012: badrecord? True
19006001.013: badrecord? True


In [30]:
# maybe a consequence of different handling of Psalm titles?
bcvref = "19005000"
print(f"For {bcvref}:")
for bcvkey in mgrnew.bcvkeys:
    print(f"in {bcvkey}? {bcvref in mgrnew.bcv[bcvkey]}")


For 19005000:
in sources? False
in targets? True
in target_sourceverses? False
in records? False
in versedata? False


Analysis: the PSA title BCV is in targets, but not sources (and therefore not in records and versedata). 
Maybe we need a version of WLCM with Psalm titles?

In [ ]:
bcvref in 

In [4]:
from difflib import Differ
d = Differ()
result = list(d.compare(list(apairs.manager1.keys()), list(apairs.manager2.keys())))
changedresults = [r for r in result if not r.startswith("  ")]
weirdindex = result.index("- ")

In [6]:
# fewer sources because of one or more AlignmentRecord instances with no sources: so grouping by BCV collapses them under ""
bcvkeys: tuple[str] = ("sources", "targets", "target_sourceverses", "records", "versedata")
for bcvkey in bcvkeys: print(f"{bcvkey:20}: {len(apairs.manager1.bcv[bcvkey])}")

sources             : 23213
targets             : 23261
target_sourceverses : 23205
records             : 23214
versedata           : 23214


In [5]:
# manager1 has a blank line between 08003012 (RUT 3:12) and 08003013
print(result[7184:7188])
print(list(apairs.manager1.keys())[7184:7188])
print(list(apairs.manager2.keys())[7184:7188])


['  08003011', '  08003012', '- ', '  08003013']
['08003011', '08003012', '', '08003013']
['08003011', '08003012', '08003013', '08003014']


In [13]:
mgrnew.badrecords["08003012.005"]

<BadRecord (08003012.005): 'No source selectors'>